# Practice Test 2 (Harder)

**120-minute timer. Same rules.**

**Structure:**
- Q1: Jacobian + gradient through attention
- Q2: Full end-to-end transformer forward pass
- Q3: Circuit analysis + SVD — 3 parts
- Q4: Advanced interpretability — 4 parts, **final part intentionally very hard**

Finish Q1–Q3 and Q4 Parts 1–3 before attempting Q4 Part 4.


In [ ]:
import numpy as np

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. softmax and check() are available.")


## Q1: Softmax Jacobian + Gradient through Attention

In [ ]:
def softmax(z, axis=-1):
    raise NotImplementedError()

def softmax_jacobian(z):
    """z: (n,). Returns (n,n) Jacobian J[i,j] = pi_i*(delta_ij - pi_j)."""
    raise NotImplementedError()

def attention_score_gradient(Q, K, V, dL_dout):
    """
    Gradient of a scalar loss L w.r.t. pre-softmax attention scores S.
    S = Q @ K.T / sqrt(d_k),  weights = softmax(S, axis=-1),  output = weights @ V.
    dL_dout: (T, d_v) — dL/d(output).
    Returns dL/dS of shape (T, T).
    Hint: dL/d(weights) = dL_dout @ V.T, shape (T,T).
          For each row t, apply softmax Jacobian to dL/d(weights[t,:]).
    """
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ1 TESTS\n" + "="*50)
# jacobian
z = np.array([0.5, -0.3, 1.2, 0.])
pi = softmax(z)
J = softmax_jacobian(z)
check("shape (4,4)", cond=lambda x: x.shape == (4,4), got=J)
check("diagonal: pi*(1-pi)", J.diagonal(), pi*(1-pi))
check("off-diag [0,1]", J[0,1], -pi[0]*pi[1])
check("rows sum to 0", J.sum(axis=1), np.zeros(4), atol=1e-10)
eps = 1e-5
J_fd = np.zeros((4,4))
for j in range(4):
    zp, zm = z.copy(), z.copy(); zp[j]+=eps; zm[j]-=eps
    J_fd[:,j] = (softmax(zp) - softmax(zm)) / (2*eps)
check("matches finite differences", J, J_fd, rtol=1e-4)
print()
# attention gradient
T, dk, dv = 4, 3, 5
np.random.seed(0)
Q = np.random.randn(T,dk); K = np.random.randn(T,dk)
V = np.random.randn(T,dv); dL = np.random.randn(T,dv)
dS = attention_score_gradient(Q, K, V, dL)
check("gradient shape (T,T)", cond=lambda x: x.shape == (T,T), got=dS)
# numerical verification
def loss_fn(S_flat):
    S = S_flat.reshape(T,T)
    return np.sum(softmax(S, axis=-1) @ V * dL)
S0 = Q @ K.T / np.sqrt(dk)
dS_num = np.zeros((T,T))
for i in range(T):
    for j in range(T):
        Sp = S0.copy(); Sm = S0.copy(); Sp[i,j]+=eps; Sm[i,j]-=eps
        dS_num[i,j] = (loss_fn(Sp) - loss_fn(Sm)) / (2*eps)
check("matches numerical gradient", dS, dS_num, rtol=1e-3, atol=1e-6)


## Q2: Full Transformer Forward Pass

In [ ]:
def gelu(x):
    raise NotImplementedError()

def layer_norm(x, eps=1e-5):
    raise NotImplementedError()

def transformer_forward(tokens, embed, unembed, layers, causal=True):
    """
    tokens: (T,) ints. embed: (vocab, dm). unembed: (dm, vocab).
    layers: list of dicts with keys W_Qs, W_Ks, W_Vs (H,dm,dk or dv), W_O (H*dv,dm), W1 (dm,dff), W2 (dff,dm).
    Steps: embed -> N x (LN->MHA->resid, LN->MLP->resid) -> final LN -> unembed.
    Returns (T, vocab_size) logits.
    """
    raise NotImplementedError()


In [ ]:
print("="*50 + "\nQ2 TESTS\n" + "="*50)
vocab, dm, H, dk, dv, dff = 50, 16, 2, 4, 4, 32
T = 8; np.random.seed(1)
tokens = np.array([3,7,12,5,22,1,9,4])
embed = np.random.randn(vocab,dm)*0.1
unembed = np.random.randn(dm,vocab)*0.1
layers = [{"W_Qs": np.random.randn(H,dm,dk)*0.1, "W_Ks": np.random.randn(H,dm,dk)*0.1,
           "W_Vs": np.random.randn(H,dm,dv)*0.1, "W_O": np.random.randn(H*dv,dm)*0.1,
           "W1": np.random.randn(dm,dff)*0.1,   "W2": np.random.randn(dff,dm)*0.1} for _ in range(2)]
logits = transformer_forward(tokens, embed, unembed, layers)
check("logits shape (T,vocab)", cond=lambda x: x.shape == (T,vocab), got=logits)
check("logits finite", cond=lambda x: np.all(np.isfinite(x)), got=logits)
check("different positions give different logits",
      cond=lambda x: not np.allclose(x[0], x[1]), got=logits)


## Q3: Circuit Analysis + SVD (3 Parts)

In [ ]:
# Part 1: Compute circuit matrices for all heads
def all_circuit_matrices(W_Qs, W_Ks, W_Vs, W_Os):
    """
    W_Qs/Ks/Vs: (H,dm,dk or dv). W_Os: (H,dv,dm) per-head output projections.
    Returns W_QKs (H,dm,dm), W_OVs (H,dm,dm).
    """
    raise NotImplementedError()


In [ ]:
print("Q3 Part 1 Tests")
H, dm, dk, dv = 4, 8, 3, 3
W_Qs = np.random.randn(H,dm,dk); W_Ks = np.random.randn(H,dm,dk)
W_Vs = np.random.randn(H,dm,dv); W_Os = np.random.randn(H,dv,dm)
W_QKs, W_OVs = all_circuit_matrices(W_Qs, W_Ks, W_Vs, W_Os)
check("W_QKs shape (H,dm,dm)", cond=lambda x: x.shape == (H,dm,dm), got=W_QKs)
check("W_OVs shape (H,dm,dm)", cond=lambda x: x.shape == (H,dm,dm), got=W_OVs)
check("W_QK[0] = W_Q[0]@W_K[0].T", W_QKs[0], W_Qs[0] @ W_Ks[0].T)
check("W_OV[0] = W_V[0]@W_O[0]",   W_OVs[0], W_Vs[0] @ W_Os[0])


In [ ]:
# Part 2: SVD of all W_OV matrices
def ov_svd_analysis(W_OVs, threshold=0.9):
    """
    W_OVs: (H,dm,dm).
    Returns:
        all_sv: (H,dm) singular values, descending
        eff_ranks: (H,) effective rank for each head at `threshold` of Frobenius norm²
    """
    raise NotImplementedError()


In [ ]:
print("Q3 Part 2 Tests")
all_sv, eff_ranks = ov_svd_analysis(W_OVs)
check("all_sv shape (H,dm)", cond=lambda x: x.shape == (H,dm), got=all_sv)
check("sv non-negative", cond=lambda x: np.all(x >= 0), got=all_sv)
check("sv descending per head", cond=lambda x: np.all(x[:,:-1] >= x[:,1:]), got=all_sv)
check("eff_ranks shape (H,)", cond=lambda x: x.shape == (H,), got=eff_ranks)
# rank ≤ min(dm, dv)=3 since W_V is (dm,dv) with dv=3
check("eff_rank ≤ min(dm,dv)=3", cond=lambda x: np.all(x <= 3), got=eff_ranks)


In [ ]:
# Part 3: Rank heads by specificity (lowest effective rank = most specific)
def rank_by_specificity(W_OVs, threshold=0.9):
    """
    Returns list of head indices sorted from most specific (lowest eff rank) to least.
    """
    raise NotImplementedError()


In [ ]:
print("Q3 Part 3 Tests")
d = 8; H_t = 4
W_OVs_t = np.zeros((H_t, d, d))
u0, v0 = np.random.randn(d), np.random.randn(d)
W_OVs_t[0] = np.outer(u0, v0)  # rank 1
W_OVs_t[1] = np.outer(np.random.randn(d), np.random.randn(d)) + np.outer(np.random.randn(d), np.random.randn(d))  # rank 2
W_OVs_t[2] = np.random.randn(d,4) @ np.random.randn(4,d)  # rank 4
W_OVs_t[3] = np.random.randn(d,d)  # full rank
ranking = rank_by_specificity(W_OVs_t)
check("all heads present", cond=lambda r: set(r) == {0,1,2,3} and len(r) == 4, got=ranking)
check("head 0 is most specific", cond=lambda r: r[0] == 0, got=ranking)
check("head 3 is least specific", cond=lambda r: r[-1] == 3, got=ranking)


## Q4: Advanced Interpretability (4 Parts)

**Complete Parts 1–3 before attempting Part 4.**
Part 4 is intentionally very hard. Don't sacrifice the earlier parts chasing it.


In [ ]:
# Part 1: Per-head entropy
def head_entropy_matrix(logits_per_layer):
    """
    logits_per_layer: list of n_layers, each a list of H arrays of shape (T,T) (pre-softmax scores).
    Returns (n_layers, H) array of mean entropy across token positions.
    entropy[l,h] = mean over t of H(softmax(logits[l][h][t,:]))
    """
    raise NotImplementedError()


In [ ]:
print("Q4 Part 1 Tests")
T, nl, H = 6, 2, 3; np.random.seed(5)
logits_all = []
for l in range(nl):
    heads = []
    for h in range(H):
        if l==0 and h==1: s = np.eye(T)*20   # peaked
        elif l==1 and h==2: s = np.zeros((T,T))  # uniform
        else: s = np.random.randn(T,T)
        heads.append(s)
    logits_all.append(heads)
E = head_entropy_matrix(logits_all)
check("shape (nl,H)", cond=lambda x: x.shape == (nl,H), got=E)
check("peaked head: low entropy", cond=lambda x: x[0,1] < 0.5, got=E)
check("uniform head: entropy ~log(T)", E[1,2], np.log(T), rtol=0.05)
check("all non-negative", cond=lambda x: np.all(x >= 0), got=E)


In [ ]:
# Part 2: Induction head scoring
def score_induction_heads(weights_per_layer, n):
    """
    weights_per_layer: list of n_layers lists of H arrays, each (2n, 2n).
    Returns (n_layers, H) induction scores in [0,1].
    """
    raise NotImplementedError()


In [ ]:
print("Q4 Part 2 Tests")
n = 5; total = 2*n; nl, H = 2, 3
np.random.seed(7)
all_w = []
for l in range(nl):
    heads = []
    for h in range(H):
        if l==1 and h==0:  # perfect induction
            w = np.zeros((total,total))
            for i in range(n-1): w[n+i, i+1] = 1.0
            w[2*n-1, :] = 1/total
            for i in range(n): w[i, :i+1] = 1/(i+1)
        else:
            w = softmax(np.random.randn(total,total), axis=-1)
        heads.append(w)
    all_w.append(heads)
sc = score_induction_heads(all_w, n)
check("shape (nl,H)", cond=lambda x: x.shape == (nl,H), got=sc)
check("planted induction scores > 0.9", cond=lambda x: x[1,0] > 0.9, got=sc)
check("scores in [0,1]", cond=lambda x: np.all((x>=0)&(x<=1)), got=sc)


In [ ]:
# Part 3: SAE decomposition
def sae_top_k(x, W_enc, b_enc, W_dec, b_dec, k=5):
    """
    x: (T,dm). Returns reconstruction (T,dm), top-k feature indices (T,k), values (T,k), MSE scalar.
    """
    raise NotImplementedError()


In [ ]:
print("Q4 Part 3 Tests")
T, dm, dh = 4, 8, 64; np.random.seed(9)
x = np.random.randn(T,dm)
W_enc = np.random.randn(dm,dh)*0.5; b_enc = np.zeros(dh)
W_dec = np.random.randn(dh,dm)*0.5; b_dec = np.zeros(dm)
recon, idx, vals, err = sae_top_k(x, W_enc, b_enc, W_dec, b_dec, k=5)
check("reconstruction shape (T,dm)", cond=lambda r: r.shape == (T,dm), got=recon)
check("indices shape (T,5)", cond=lambda i: i.shape == (T,5), got=idx)
check("values non-negative", cond=lambda v: np.all(v >= 0), got=vals)
check("values descending", cond=lambda v: np.all(v[:,:-1] >= v[:,1:]), got=vals)
check("MSE scalar >= 0", cond=lambda e: np.array(e).ndim == 0 and e >= 0, got=err)


In [ ]:
# Part 4 (HARD): Self-reinforcing circuits
#
# A virtual head is formed by composing L1 head h1 and L2 head h2:
#   W_OV_virtual = W_OVs_L2[h2] @ W_OVs_L1[h1]
#
# Feature f "self-reinforces" through this virtual head if:
#   W_OV_virtual maps W_dec[f,:] (the feature direction) back toward W_enc[:,f]
# i.e., alignment = W_enc[:,f] @ (W_OV_virtual @ W_dec[f,:])
#
# Find top-k (h1, h2, feature) triples by alignment score.

def find_self_reinforcing(W_OVs_L1, W_OVs_L2, W_enc, W_dec, top_k=3):
    """
    W_OVs_L1: (H1,dm,dm). W_OVs_L2: (H2,dm,dm).
    W_enc: (dm,dh). W_dec: (dh,dm).
    Returns list of (h1, h2, feature, score) sorted descending by score.
    """
    raise NotImplementedError()


In [ ]:
print("Q4 Part 4 Tests (HARD)")
H1, H2, dm, dh = 3, 3, 8, 32; np.random.seed(42)
W_OVs_L1 = np.random.randn(H1,dm,dm)*0.3
W_OVs_L2 = np.random.randn(H2,dm,dm)*0.3
W_enc_s = np.random.randn(dm,dh); W_dec_s = np.random.randn(dh,dm)
# Plant: h1=0, h2=1, feature=5 — make virtual OV ≈ 5*I and feature 5 live in a fixed direction
feat_dir = np.random.randn(dm); feat_dir /= np.linalg.norm(feat_dir)
W_enc_s[:,5] = feat_dir*10; W_dec_s[5,:] = feat_dir
W_OVs_L1[0] = np.eye(dm)*2; W_OVs_L2[1] = np.eye(dm)*2.5
results = find_self_reinforcing(W_OVs_L1, W_OVs_L2, W_enc_s, W_dec_s, top_k=3)
check("returns 3 tuples", cond=lambda r: len(r) == 3, got=results)
check("each tuple has 4 elements", cond=lambda r: all(len(t)==4 for t in r), got=results)
h1, h2, feat, score = results[0]
check("planted h1=0 is top", h1, 0)
check("planted h2=1 is top", h2, 1)
check("planted feature=5 is top", feat, 5)
check("sorted descending", cond=lambda r: r[0][3] >= r[1][3] >= r[2][3], got=results)
